# GulfDealFlow — Script 02: News Scraper

Pulls GCC funding news from multiple RSS sources and extracts deal data into a staging CSV for your review.

**How it works:**
- Tries multiple news RSS feeds
- Extracts deal signals (amounts, stages, countries) from headlines
- Saves everything to `news_staging.csv` in your GulfDealFlow Drive folder
- You review the staging file, fill gaps, delete non-deals, then run Script 03

⚠️ If all sources return 403, download this notebook and run it locally.

In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2 — Install dependencies
!pip install requests feedparser pandas -q
print('✓ Dependencies installed')

In [ ]:
# CELL 3 — Setup
import os
import re
import time
import requests
import feedparser
import pandas as pd
from datetime import datetime

BASE_DIR    = '/content/drive/MyDrive/GulfDealFlow'
OUTPUT_PATH = os.path.join(BASE_DIR, 'news_staging.csv')
os.makedirs(BASE_DIR, exist_ok=True)
print(f'✓ Working directory ready: {BASE_DIR}')

In [ ]:
# CELL 4 — Config: sources, patterns, mappings

# RSS feeds to try — ordered by reliability
RSS_FEEDS = [
    {
        'name': 'Google News — UAE funding',
        'url': 'https://news.google.com/rss/search?q=startup+funding+UAE+million&hl=en&gl=AE&ceid=AE:en'
    },
    {
        'name': 'Google News — Saudi funding',
        'url': 'https://news.google.com/rss/search?q=startup+funding+Saudi+Arabia+million&hl=en&gl=SA&ceid=SA:en'
    },
    {
        'name': 'Google News — GCC venture',
        'url': 'https://news.google.com/rss/search?q=venture+capital+GCC+raises+million&hl=en&gl=US&ceid=US:en'
    },
    {
        'name': 'Google News — Series A GCC',
        'url': 'https://news.google.com/rss/search?q=series+a+funding+UAE+Saudi+startup&hl=en&gl=US&ceid=US:en'
    },
    {
        'name': 'Google News — seed GCC 2024',
        'url': 'https://news.google.com/rss/search?q=seed+funding+GCC+startup+2024&hl=en&gl=US&ceid=US:en'
    },
    {
        'name': 'Google News — seed GCC 2025',
        'url': 'https://news.google.com/rss/search?q=seed+funding+GCC+startup+2025&hl=en&gl=US&ceid=US:en'
    },
    {
        'name': 'Google News — raises million Gulf',
        'url': 'https://news.google.com/rss/search?q=raises+million+Gulf+startup+investment&hl=en&gl=US&ceid=US:en'
    },
    {
        'name': 'TechCrunch Middle East',
        'url': 'https://techcrunch.com/tag/middle-east/feed/'
    },
]

# Country detection
CITY_TO_COUNTRY = {
    'dubai':     ('UAE', 'Dubai'),
    'abu dhabi': ('UAE', 'Abu Dhabi'),
    'sharjah':   ('UAE', 'Sharjah'),
    'riyadh':    ('Saudi Arabia', 'Riyadh'),
    'jeddah':    ('Saudi Arabia', 'Jeddah'),
    'neom':      ('Saudi Arabia', 'NEOM'),
    'muscat':    ('Oman', 'Muscat'),
    'doha':      ('Qatar', 'Doha'),
    'manama':    ('Bahrain', 'Manama'),
    'kuwait city': ('Kuwait', 'Kuwait City'),
}

COUNTRY_NAMES = {
    'uae':           'UAE',
    'united arab emirates': 'UAE',
    'saudi arabia':  'Saudi Arabia',
    'ksa':           'Saudi Arabia',
    'kuwait':        'Kuwait',
    'bahrain':       'Bahrain',
    'oman':          'Oman',
    'qatar':         'Qatar',
}

STAGE_PATTERNS = [
    ('Pre-Seed',  r'\bpre[\-\s]?seed\b'),
    ('Seed',      r'\bseed\s+(round|funding|stage)\b'),
    ('Series A',  r'\bseries\s+a\b'),
    ('Series B',  r'\bseries\s+b\b'),
    ('Series C+', r'\bseries\s+[cdefg]\b'),
    ('Growth',    r'\bgrowth\s+(round|equity)\b|\blate[\-\s]stage\b'),
]

SECTOR_KEYWORDS = {
    'Fintech':                   ['fintech', 'payment', 'neobank', 'lending', 'insurtech', 'crypto', 'remittance', 'bnpl', 'buy now pay later'],
    'Proptech':                  ['proptech', 'real estate tech', 'property tech'],
    'Logistics & Supply Chain':  ['logistics', 'supply chain', 'last mile', 'freight', 'delivery', 'fleet'],
    'Healthtech':                ['healthtech', 'telehealth', 'medtech', 'digital health', 'pharmatech'],
    'Edtech':                    ['edtech', 'ed tech', 'education tech', 'e-learning', 'online learning'],
    'E-commerce & Retail':       ['e-commerce', 'ecommerce', 'marketplace', 'retail tech', 'd2c'],
    'SaaS & Enterprise Software':['saas', 'enterprise software', 'b2b software', 'cloud platform'],
    'Deep Tech & AI':            ['ai startup', 'artificial intelligence', 'machine learning', 'deep tech', 'robotics'],
    'Energy & Cleantech':        ['cleantech', 'clean energy', 'solar', 'renewable', 'green tech'],
    'Media & Entertainment':     ['media tech', 'streaming', 'gaming', 'content platform'],
    'Food & Agritech':           ['food tech', 'agritech', 'restaurant tech', 'cloud kitchen'],
}

FUNDING_SIGNALS = ['raises', 'raised', 'secures', 'secured', 'closes', 'closed',
                   'funding', 'series', 'seed', 'million', 'billion', 'investment', 'backed']

print('✓ Config loaded')

In [ ]:
# CELL 5 — Helper functions

def get_feed(url, name):
    """Fetch RSS feed using feedparser with fallback to requests."""
    try:
        # Try feedparser first
        feed = feedparser.parse(url)
        if feed.entries:
            print(f'  ✓ {name}: {len(feed.entries)} articles (feedparser)')
            return feed.entries

        # Feedparser got nothing — try raw requests
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'application/rss+xml, application/xml, text/xml, */*',
        }
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code == 200:
            feed = feedparser.parse(r.text)
            if feed.entries:
                print(f'  ✓ {name}: {len(feed.entries)} articles (requests fallback)')
                return feed.entries
        print(f'  ✗ {name}: status {r.status_code}, no entries')
        return []

    except Exception as e:
        print(f'  ✗ {name}: {str(e)[:60]}')
        return []


def extract_amount(text):
    """Pull funding amount from text, return (int USD, disclosed bool)."""
    t = text.lower()
    patterns = [
        (r'\$(\d+(?:\.\d+)?)\s*billion',               1_000_000_000),
        (r'\$(\d+(?:\.\d+)?)\s*million',               1_000_000),
        (r'\$(\d+(?:\.\d+)?)m\b',                      1_000_000),
        (r'(\d+(?:\.\d+)?)\s*billion\s*(?:dollar|usd)', 1_000_000_000),
        (r'(\d+(?:\.\d+)?)\s*million\s*(?:dollar|usd)', 1_000_000),
        (r'aed\s*(\d+(?:\.\d+)?)\s*(?:billion)',        1_000_000_000 / 3.67),
        (r'aed\s*(\d+(?:\.\d+)?)\s*(?:million)',        1_000_000 / 3.67),
        (r'sar\s*(\d+(?:\.\d+)?)\s*(?:million)',        1_000_000 / 3.75),
    ]
    for pattern, multiplier in patterns:
        m = re.search(pattern, t)
        if m:
            try:
                return int(float(m.group(1)) * multiplier), True
            except:
                continue
    return None, False


def extract_stage(text):
    t = text.lower()
    for stage, pattern in STAGE_PATTERNS:
        if re.search(pattern, t):
            return stage
    return 'Undisclosed'


def extract_country_city(text):
    t = text.lower()
    # Check cities first (more specific)
    for city_key, (country, city) in CITY_TO_COUNTRY.items():
        if city_key in t:
            return country, city
    # Then country names
    for name, country in COUNTRY_NAMES.items():
        if name in t:
            return country, ''
    return None, ''


def extract_sector(text):
    t = text.lower()
    for sector, keywords in SECTOR_KEYWORDS.items():
        for kw in keywords:
            if kw in t:
                return sector
    return 'Other'


def extract_company(title):
    """Pull company name from common headline patterns."""
    patterns = [
        r'^([A-Z][a-zA-Z0-9\s\-\.]+?)\s+(?:raises?|secures?|closes?|lands?|gets?|bags?)\s',
        r'^([A-Z][a-zA-Z0-9\s\-\.]+?),\s+(?:a|an|the)\s+',
        r'(?:funding for|investment in|backs?)\s+([A-Z][a-zA-Z0-9\s\-\.]+)',
    ]
    for pat in patterns:
        m = re.search(pat, title)
        if m:
            name = m.group(1).strip()
            # Sanity check — reject if too long or looks like a sentence
            if len(name) < 40 and name.count(' ') < 5:
                return name
    return ''  # blank = needs manual review


def parse_date(entry):
    """Try multiple date fields from RSS entry."""
    for field in ['published', 'updated', 'created']:
        raw = getattr(entry, field, None)
        if raw:
            for fmt in ['%a, %d %b %Y %H:%M:%S %z', '%a, %d %b %Y %H:%M:%S GMT',
                        '%Y-%m-%dT%H:%M:%S%z', '%Y-%m-%d']:
                try:
                    return datetime.strptime(raw[:25].strip(), fmt).strftime('%Y-%m')
                except:
                    continue
            # Last resort: grab YYYY-MM if present
            m = re.search(r'(\d{4})-(\d{2})', raw)
            if m:
                return f'{m.group(1)}-{m.group(2)}'
    return ''


print('✓ Helper functions ready')

In [ ]:
# CELL 6 — Run the scraper

print('GulfDealFlow News Scraper\n' + '='*40)

all_deals = []
seen_titles = set()
sources_hit = 0

for feed_config in RSS_FEEDS:
    print(f"\nFetching: {feed_config['name']}")
    entries = get_feed(feed_config['url'], feed_config['name'])

    if not entries:
        continue

    sources_hit += 1
    deals_this_source = 0

    for entry in entries:
        # Get title and summary safely
        title   = getattr(entry, 'title', '') or ''
        summary = getattr(entry, 'summary', '') or ''
        link    = getattr(entry, 'link', '') or ''
        title   = re.sub(r'<[^>]+>', '', title).strip()    # strip any HTML tags
        summary = re.sub(r'<[^>]+>', '', summary).strip()

        # Deduplicate
        if title in seen_titles or not title:
            continue
        seen_titles.add(title)

        # Check for funding signals
        full_text = (title + ' ' + summary).lower()
        if not any(sig in full_text for sig in FUNDING_SIGNALS):
            continue

        # Country filter — must be GCC
        country, city = extract_country_city(full_text)
        if not country:
            continue

        # Extract deal fields
        amount, disclosed = extract_amount(full_text)
        stage             = extract_stage(full_text)
        sector            = extract_sector(full_text)
        company           = extract_company(title)
        date_str          = parse_date(entry)

        all_deals.append({
            'deal_id':        '',
            'company_name':   company,
            'country':        country,
            'city':           city,
            'date':           date_str,
            'stage':          stage,
            'amount_usd':     amount if amount else '',
            'disclosed':      'TRUE' if disclosed else 'FALSE',
            'sector':         sector,
            'description':    '',
            'founded_year':   '',
            'website':        '',
            'lead_investor':  '',
            'co_investors':   '',
            'investor_types': '',
            'source':         'News — ' + feed_config['name'],
            'notes':          f'REVIEW | {title[:100]} | {link[:80]}',
        })
        deals_this_source += 1

    print(f'  → {deals_this_source} potential deals extracted')
    time.sleep(1.5)

# ── Output ──────────────────────────────────────────────────────
print('\n' + '='*40)

if not all_deals:
    if sources_hit == 0:
        print('⚠ All RSS feeds blocked — Colab IP is being rate-limited.')
        print('  Download this notebook and run it locally instead.')
        print('  Everything else (Scripts 01 and 03) can stay on Colab.')
    else:
        print('⚠ Feeds loaded but no GCC funding deals detected.')
        print('  Try again later or adjust the RSS queries in CELL 4.')
else:
    df = pd.DataFrame(all_deals)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f'✓ {len(df)} potential deals extracted from {sources_hit} source(s)')
    print(f'✓ Saved: {OUTPUT_PATH}')
    blank_names = (df['company_name'] == '').sum()
    print(f'\n  Deals needing company name review: {blank_names}')
    print(f'  Amount disclosed: {(df["disclosed"]=="TRUE").sum()} / {len(df)}')
    print(f'\n⚠ Next step: open news_staging.csv in Drive, review each row.')
    print('  Use the notes column to see the original headline.')
    print('  Fill blanks, delete non-deals, then run Script 03.')

In [ ]:
# CELL 7 — Preview staging file
if all_deals:
    df[['company_name','country','date','stage','amount_usd','sector','notes']].head(15)
else:
    print('No deals to preview.')